## pdf 다운로드/파싱

In [1]:
import requests
import fitz
import numpy as np
from pathlib import Path

BASE_DIR = Path.cwd()

def pdf_download(url:str, path:str) -> bool:
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/pdf,*/*",
    }
    
    if url == "":
        return False
    
    res = requests.get(url, headers=headers, timeout=30)

    if res.status_code == 200 and "pdf" in res.headers.get("content-type", "").lower():
        with open(path, "wb") as f:
            f.write(res.content)
        return True
    else:
        return False

def pdf_read(path:str) -> list[str]:
    ls = path.split("/")
    
    path_data = BASE_DIR
    for l in ls:
        path_data = path_data / l
    
    docs = fitz.open(path_data)

    buff = []
    for page in docs:
        buff.append(str(page.get_text()).replace("\n", " "))
    
    return buff

In [2]:
import pandas as pd
import time

In [3]:
# 원본 데이터 csv 경로
source_csv_path = "data\database\ProductTV.csv"

# pdf 파일들 생성될 경로
pdf_path = "manual_pdf/manual_TV_pdf"

# 설명서 쪼갤 길이 단위
token_len = 100

# 결과 csv 경로
result_path = "res/res_TV.csv"

<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Playdata\AppData\Local\Temp\ipykernel_21320\1481800592.py:2: SyntaxWarning: invalid escape sequence '\d'
  source_csv_path = "data\database\ProductTV.csv"


In [4]:
import pandas as pd
import time
from tqdm import tqdm

# 데이터 csv 로드
source = pd.read_csv(source_csv_path).replace(np.nan, '')

# 필요 컬럼 추출
codes = source["product_code"].to_list()
links = source["manual_link"].to_list()

# 순차적으로 다운로드
buff_failed = []

# tqdm을 pbar라는 변수로 지정해 줍니다.
pbar = tqdm(range(len(codes)), ncols=100)

for i in pbar:
    # 💡 꿀팁: 진행 바 왼쪽에 현재 진행 중인 작업(코드명)을 실시간으로 띄워줍니다!
    pbar.set_description(f"다운로드 중: {codes[i]}")
    
    p = pdf_path + f"/{codes[i]}.pdf"

    # 🚨 주의: pdf_download() 내부의 print()문은 반드시 지워져 있어야 합니다!
    if not pdf_download(links[i], p):
        buff_failed.append(i)

    time.sleep(1)

# 결과 출력
print(f"\n\n{len(buff_failed)}개 pdf 다운로드 실패\n")
for f_ind in buff_failed:
    print(f"{f_ind + 1}번째 pdf\t코드: {codes[f_ind]}\t링크: {links[f_ind]}")

다운로드 중: TVT190: 100%|████████████████████████████████████████| 191/191 [05:18<00:00,  1.67s/it]it/s]



2개 pdf 다운로드 실패

78번째 pdf	코드: TVT077	링크: 
95번째 pdf	코드: TVT094	링크: 


In [5]:
# pdf 순차적으로 읽어들임
buff_code = []
buff_page_num = []
buff_contents = []
for i in range(len(codes)):
    p = pdf_path + f"\\{codes[i]}.pdf"

    # 페이지별 내용 읽어오기
    try:
        pages = pdf_read(p)
    except Exception as e:
        print(p + "읽기 실패" + str(e))
        continue

    # 페이지별로 나누기
    for j in range(len(pages)):
        page_text = pages[j]

        # token_len 기준으로 문자열 자르기
        for k in range(0, len(page_text), token_len):
            chunk = page_text[k:k + token_len]

            # 데이터 저장
            buff_code.append(codes[i])
            buff_page_num.append(j + 1)
            buff_contents.append(chunk)

manual_pdf/manual_TV_pdf\TVT077.pdf읽기 실패no such file: 'c:\lecture\4th_project\products\manual_pdf\manual_TV_pdf\TVT077.pdf'
manual_pdf/manual_TV_pdf\TVT094.pdf읽기 실패no such file: 'c:\lecture\4th_project\products\manual_pdf\manual_TV_pdf\TVT094.pdf'


In [6]:
# 얻은 데이터 dataframe 로드
res_df = pd.DataFrame({
    "code": buff_code,
    "page": buff_page_num,
    "content": buff_contents
})

# dataframe > csv
res_df.to_csv(result_path, index=False)